/Users/YourUserName123/Downloads/Internship_SCIPE CI-SIP/MainProject/2_Job/BLS_Job_Data/raw/occupation_projection_2024_2034.xlsx
https://www.bls.gov/emp/ind-occ-matrix/occupation.xlsx

# Fail

In [ ]:
# ============================================================
# BLS JOB DATA PROJECT - DOWNLOAD ONLY
# FIXED VERSION
#
# Downloads:
# 1. Projection / Forecast:
#    https://www.bls.gov/emp/ind-occ-matrix/occupation.xlsx
#
# 2. OEWS wage history:
#    Main source page:
#    https://www.bls.gov/oes/tables.htm
#
# FIX:
# - If BLS blocks Python from scraping tables.htm with HTTP 403,
#   this code DOES NOT STOP.
# - It automatically builds official BLS direct download URLs.
#
# IMPORTANT:
# - This code DOES NOT use:
#   BLS_Job_Data/output/oews_wage_download_links.csv
#
# MEMORY OPTIMIZED:
# - Downloads with stream=True
# - Uses 1 MB chunks
# - Skips existing files
# - Saves .part temporary file first
# - Does not load all files into RAM
# ============================================================

from pathlib import Path
from urllib.parse import urljoin, urlparse
from html import unescape
import requests
import csv
import time
import re
import os

# ============================================================
# 0. SETTINGS
# ============================================================

start_time = time.time()

PROJECT_DIR = Path.cwd()

BASE_DIR = PROJECT_DIR / "BLS_Job_Data"
RAW_DIR = BASE_DIR / "raw"
OUTPUT_DIR = BASE_DIR / "output"

PROJECTION_DIR = RAW_DIR / "projection_forecast"
WAGE_DIR = RAW_DIR / "oews_wage_history"
SOURCE_PAGE_DIR = RAW_DIR / "source_pages"

PROJECTION_DIR.mkdir(parents=True, exist_ok=True)
WAGE_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_PAGE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Projection / forecast file
PROJECTION_URL = "https://www.bls.gov/emp/ind-occ-matrix/occupation.xlsx"
PROJECTION_FILE = PROJECTION_DIR / "occupation_projection_2024_2034.xlsx"

# OEWS wage source page
OEWS_TABLES_URL = "https://www.bls.gov/oes/tables.htm"
OEWS_1988_1995_URL = "https://www.bls.gov/oes/estimates_88_95.htm"

# Output logs
SCRAPED_LINKS_FILE = OUTPUT_DIR / "oews_wage_links_scraped_or_generated.csv"
MANIFEST_FILE = OUTPUT_DIR / "bls_download_only_manifest.csv"

# Choose what to download:
# "all_files" = National + State + Metro + Industry + All Data + old 1988-1995 files
# "all_data_only" = only files labeled All Data where available
DOWNLOAD_MODE = "all_files"

CHUNK_SIZE = 1024 * 1024      # 1 MB chunks = memory friendly
TIMEOUT = 60
MAX_RETRIES = 3
SLEEP_BETWEEN_DOWNLOADS = 0.5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/605.1.15 (KHTML, like Gecko) "
        "Version/17.0 Safari/605.1.15"
    ),
    "Accept": "*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.bls.gov/",
    "Connection": "keep-alive",
}

HTML_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/605.1.15 (KHTML, like Gecko) "
        "Version/17.0 Safari/605.1.15"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.bls.gov/",
    "Connection": "keep-alive",
}

print("=" * 80)
print("BLS DOWNLOAD ONLY PROJECT - FIXED 403 VERSION")
print("=" * 80)
print("Current Jupyter folder:")
print(PROJECT_DIR)
print()
print("Projection / forecast will save here:")
print(PROJECTION_DIR)
print()
print("OEWS wage history will save here:")
print(WAGE_DIR)
print()
print("Output logs will save here:")
print(OUTPUT_DIR)
print()
print("Download mode:", DOWNLOAD_MODE)
print()

# ============================================================
# 1. HELPER FUNCTIONS
# ============================================================

def elapsed():
    return round(time.time() - start_time, 2)

def clean_text(text):
    text = unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def clean_filename(text):
    text = str(text).strip()
    text = text.replace("https://", "").replace("http://", "")
    text = text.replace("–", "-")
    text = text.replace("—", "-")
    text = re.sub(r"[^\w\-.]+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_")[:180]

def get_file_extension(url):
    path = urlparse(url).path.lower()

    for ext in [".xlsx", ".xls", ".txt", ".zip", ".csv"]:
        if path.endswith(ext):
            return ext

    return ""

def is_download_file(url, link_text=""):
    ext = get_file_extension(url)

    if ext in [".xlsx", ".xls", ".txt", ".zip", ".csv"]:
        return True

    text = str(link_text).lower().strip()

    if text in ["xlsx", "xls", "txt", "zip", "csv"]:
        return True

    return False

def guess_year_or_period(text):
    text = str(text)

    range_match = re.search(r"(19\d{2}|20\d{2})\s*[-–]\s*(19\d{2}|20\d{2})", text)
    if range_match:
        return f"{range_match.group(1)}_{range_match.group(2)}"

    year_match = re.search(r"(19\d{2}|20\d{2})", text)
    if year_match:
        return year_match.group(1)

    return "unknown_year"

def get_url_filename(url, fallback="download"):
    name = Path(urlparse(url).path).name

    if not name:
        name = fallback

    name = clean_filename(name)

    if "." not in name:
        ext = get_file_extension(url)
        if ext:
            name += ext
        else:
            name += ".dat"

    return name

def make_download_filename(row, used_names):
    period = clean_filename(row["period"])
    category = clean_filename(row["category"])
    source_name = get_url_filename(row["url"], "oews_download")

    filename = f"{period}_{category}_{source_name}"
    filename = clean_filename(filename)

    stem = Path(filename).stem
    suffix = Path(filename).suffix

    if not suffix:
        suffix = ".dat"

    final_name = stem + suffix
    counter = 2

    while final_name in used_names:
        final_name = f"{stem}_{counter}{suffix}"
        counter += 1

    used_names.add(final_name)
    return final_name

def file_is_good(path, min_size=1000):
    path = Path(path)
    return path.exists() and path.stat().st_size >= min_size

# ============================================================
# 2. HTML FETCH THAT DOES NOT CRASH ON 403
# ============================================================

def fetch_html_no_crash(url, local_backup_path=None):
    """
    Tries to read BLS HTML page.

    If BLS returns 403, this returns None instead of stopping.
    Then the code will use generated official BLS direct URLs.
    """

    print(f"[{elapsed()}s] Trying to read source page:")
    print(url)

    if local_backup_path is None:
        local_backup_path = SOURCE_PAGE_DIR / clean_filename(Path(urlparse(url).path).name or "source_page.html")

    local_backup_path = Path(local_backup_path)

    # Use local backup first if it exists
    if local_backup_path.exists() and local_backup_path.stat().st_size > 1000:
        print(f"[{elapsed()}s] Using local HTML backup:")
        print(local_backup_path)
        print()
        return local_backup_path.read_text(encoding="utf-8", errors="ignore")

    try:
        response = requests.get(
            url,
            headers=HTML_HEADERS,
            timeout=TIMEOUT
        )

        if response.status_code == 200 and response.text.strip():
            html = response.text
            local_backup_path.write_text(html, encoding="utf-8")
            print(f"[{elapsed()}s] Source page read successfully.")
            print("Saved HTML backup:")
            print(local_backup_path)
            print()
            return html

        print(f"[{elapsed()}s] Could not read page. HTTP status:", response.status_code)
        print("Will use generated official BLS direct URLs instead.")
        print()
        return None

    except Exception as e:
        print(f"[{elapsed()}s] Could not read page:", e)
        print("Will use generated official BLS direct URLs instead.")
        print()
        return None

# ============================================================
# 3. DOWNLOAD FUNCTION - MEMORY OPTIMIZED
# ============================================================

def download_file(url, save_path, label=""):
    """
    Memory optimized:
    - streams file in chunks
    - does not load whole file into RAM
    - saves to .part temporary file first
    - skips file if already exists
    - retries failed download
    """

    save_path = Path(save_path)
    temp_path = save_path.with_suffix(save_path.suffix + ".part")

    if file_is_good(save_path):
        print(f"[{elapsed()}s] SKIP already exists: {save_path.name}")

        return {
            "status": "skipped_exists",
            "url": url,
            "file": str(save_path),
            "size_bytes": save_path.stat().st_size,
            "error": ""
        }

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(f"[{elapsed()}s] Downloading {label} attempt {attempt}/{MAX_RETRIES}")
            print("URL :", url)
            print("SAVE:", save_path)

            with requests.get(
                url,
                headers=HEADERS,
                stream=True,
                timeout=TIMEOUT
            ) as response:

                if response.status_code != 200:
                    raise RuntimeError(f"HTTP status code {response.status_code}")

                total_size = int(response.headers.get("content-length", 0))
                downloaded = 0
                last_print_mb = 0

                with open(temp_path, "wb") as f:
                    for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                        if chunk:
                            f.write(chunk)
                            downloaded += len(chunk)

                            downloaded_mb = downloaded / 1024 / 1024

                            if downloaded_mb - last_print_mb >= 10:
                                if total_size:
                                    pct = downloaded / total_size * 100
                                    print(f"    {downloaded_mb:.1f} MB downloaded ({pct:.1f}%)")
                                else:
                                    print(f"    {downloaded_mb:.1f} MB downloaded")

                                last_print_mb = downloaded_mb

                if not temp_path.exists() or temp_path.stat().st_size < 1:
                    raise RuntimeError("Downloaded temp file is empty.")

                os.replace(temp_path, save_path)

                size_bytes = save_path.stat().st_size

                print(f"[{elapsed()}s] DONE: {save_path.name}")
                print(f"Size: {size_bytes / 1024 / 1024:.2f} MB")
                print()

                return {
                    "status": "downloaded",
                    "url": url,
                    "file": str(save_path),
                    "size_bytes": size_bytes,
                    "error": ""
                }

        except Exception as e:
            print(f"[{elapsed()}s] ERROR attempt {attempt}: {e}")

            if temp_path.exists():
                try:
                    temp_path.unlink()
                except Exception:
                    pass

            if attempt < MAX_RETRIES:
                time.sleep(3)

            else:
                print(f"[{elapsed()}s] FAILED after retries")
                print()

                return {
                    "status": "failed",
                    "url": url,
                    "file": str(save_path),
                    "size_bytes": 0,
                    "error": str(e)
                }

# ============================================================
# 4. SCRAPE OEWS LINKS FROM HTML WHEN POSSIBLE
# ============================================================

def extract_download_links_from_html(html, page_url, page_label):
    """
    Extracts direct download links from a BLS OEWS page.

    Tracks headers like:
    May 2025
    May 2024
    1995
    1994
    """

    results = []
    current_period = "unknown_period"

    token_pattern = re.compile(
        r"(<h1[^>]*>.*?</h1>|<h2[^>]*>.*?</h2>|<h3[^>]*>.*?</h3>|<h4[^>]*>.*?</h4>|<li[^>]*>.*?</li>|<tr[^>]*>.*?</tr>)",
        flags=re.IGNORECASE | re.DOTALL
    )

    href_pattern = re.compile(
        r'<a[^>]+href=["\']([^"\']+)["\'][^>]*>(.*?)</a>',
        flags=re.IGNORECASE | re.DOTALL
    )

    for match in token_pattern.finditer(html):
        token = match.group(1)
        token_lower = token.lower()

        if (
            token_lower.startswith("<h1")
            or token_lower.startswith("<h2")
            or token_lower.startswith("<h3")
            or token_lower.startswith("<h4")
        ):
            header_text = clean_text(token)

            if re.search(r"(19\d{2}|20\d{2})", header_text):
                current_period = header_text

            continue

        parent_text = clean_text(token)

        for href, link_html in href_pattern.findall(token):
            link_text = clean_text(link_html)
            full_url = urljoin(page_url, href)

            if not is_download_file(full_url, link_text):
                continue

            category = parent_text

            category = re.sub(r"\bHTML\b", "", category, flags=re.IGNORECASE)
            category = re.sub(r"\bXLSX\b", "", category, flags=re.IGNORECASE)
            category = re.sub(r"\bXLS\b", "", category, flags=re.IGNORECASE)
            category = re.sub(r"\bTXT\b", "", category, flags=re.IGNORECASE)
            category = re.sub(r"\bZIP\b", "", category, flags=re.IGNORECASE)
            category = re.sub(r"\bCSV\b", "", category, flags=re.IGNORECASE)
            category = re.sub(r"\(\s*\)", "", category)
            category = re.sub(r"\s+", " ", category).strip()

            if not category:
                category = link_text

            if DOWNLOAD_MODE == "all_data_only":
                if "all data" not in category.lower():
                    continue

            results.append({
                "page_label": page_label,
                "period": current_period,
                "year_or_period": guess_year_or_period(current_period),
                "category": category,
                "link_text": link_text,
                "url": full_url,
                "link_source": "scraped_from_bls_html"
            })

    return results

# ============================================================
# 5. GENERATED FALLBACK OFFICIAL BLS DIRECT URL LIST
# ============================================================

def add_generated_link(rows, period, category, url, link_text="generated"):
    """
    Adds a generated official BLS URL.
    Applies DOWNLOAD_MODE filter.
    """

    if DOWNLOAD_MODE == "all_data_only":
        if "all data" not in category.lower():
            return

    rows.append({
        "page_label": "generated fallback official BLS direct URL",
        "period": period,
        "year_or_period": guess_year_or_period(period),
        "category": category,
        "link_text": link_text,
        "url": url,
        "link_source": "generated_fallback_direct_url"
    })

def build_generated_oews_links():
    """
    Builds official BLS OEWS/OES direct download URLs.

    This is used when https://www.bls.gov/oes/tables.htm blocks Python with 403.

    Main patterns:
    - 2025 to 2003 May files: oesmYY...
    - November 2003 and November 2004: oesnYY...
    - 2002 to 1997 older files: oesYY...
    - 1988 to 1995 historical XLS files: special request XLS files
    """

    rows = []
    base = "https://www.bls.gov/oes/special-requests/"

    def add_modern_set(year, prefix, include_all=False, industry_suffix="in4", period_prefix="May"):
        yy = str(year)[-2:]
        period = f"{period_prefix} {year}"

        add_generated_link(
            rows,
            period,
            "National",
            f"{base}{prefix}nat.zip",
            "ZIP"
        )

        add_generated_link(
            rows,
            period,
            "State",
            f"{base}{prefix}st.zip",
            "ZIP"
        )

        add_generated_link(
            rows,
            period,
            "Metropolitan and nonmetropolitan area",
            f"{base}{prefix}ma.zip",
            "ZIP"
        )

        add_generated_link(
            rows,
            period,
            "National industry-specific and by ownership",
            f"{base}{prefix}{industry_suffix}.zip",
            "ZIP"
        )

        if include_all:
            add_generated_link(
                rows,
                period,
                "All data",
                f"{base}{prefix}all.zip",
                "ZIP"
            )

    # May 2025 through May 2014
    # These have National, State, Metro, Industry, All Data
    for year in range(2025, 2013, -1):
        yy = str(year)[-2:]
        add_modern_set(
            year=year,
            prefix=f"oesm{yy}",
            include_all=True,
            industry_suffix="in4",
            period_prefix="May"
        )

    # May 2013 and May 2012 also have All Data
    for year in [2013, 2012]:
        yy = str(year)[-2:]
        add_modern_set(
            year=year,
            prefix=f"oesm{yy}",
            include_all=True,
            industry_suffix="in4",
            period_prefix="May"
        )

    # November 2011 agricultural file
    # It is not "All data", so it is skipped when DOWNLOAD_MODE = all_data_only
    add_generated_link(
        rows,
        "November 2011",
        "Agricultural Sector Estimates",
        "https://www.bls.gov/oes/ag_data_2011.xlsx",
        "XLSX"
    )

    # May 2011 has All Data
    add_modern_set(
        year=2011,
        prefix="oesm11",
        include_all=True,
        industry_suffix="in4",
        period_prefix="May"
    )

    # May 2010 through May 2005
    # These do not list All Data on the main page
    for year in range(2010, 2004, -1):
        yy = str(year)[-2:]
        add_modern_set(
            year=year,
            prefix=f"oesm{yy}",
            include_all=False,
            industry_suffix="in4",
            period_prefix="May"
        )

    # May 2004 and November 2004
    add_modern_set(
        year=2004,
        prefix="oesm04",
        include_all=False,
        industry_suffix="in4",
        period_prefix="May"
    )

    add_modern_set(
        year=2004,
        prefix="oesn04",
        include_all=False,
        industry_suffix="in4",
        period_prefix="November"
    )

    # May 2003 and November 2003
    add_modern_set(
        year=2003,
        prefix="oesm03",
        include_all=False,
        industry_suffix="in4",
        period_prefix="May"
    )

    add_modern_set(
        year=2003,
        prefix="oesn03",
        include_all=False,
        industry_suffix="in4",
        period_prefix="November"
    )

    # 2002 uses oes02 and industry in4
    add_modern_set(
        year=2002,
        prefix="oes02",
        include_all=False,
        industry_suffix="in4",
        period_prefix=""
    )

    # 2001 to 1997 use oesYY and industry in3
    for year in range(2001, 1996, -1):
        yy = str(year)[-2:]
        add_modern_set(
            year=year,
            prefix=f"oes{yy}",
            include_all=False,
            industry_suffix="in3",
            period_prefix=""
        )

    # 1988 to 1995 historical XLS files
    # 2-digit and 3-digit SIC level
    historical_prefixes = {
        1995: "mf95",
        1994: "bn94",
        1993: "nm93",
        1992: "mf92",
        1991: "bn91",
        1990: "nm90",
        1989: "mf89",
        1988: "bn88",
    }

    for year in sorted(historical_prefixes.keys(), reverse=True):
        prefix = historical_prefixes[year]

        add_generated_link(
            rows,
            str(year),
            "Historical national industry-specific 2-digit SIC",
            f"{base}{prefix}d2.xls",
            "XLS"
        )

        add_generated_link(
            rows,
            str(year),
            "Historical national industry-specific 3-digit SIC",
            f"{base}{prefix}d3.xls",
            "XLS"
        )

    return rows

# ============================================================
# 6. STEP 1 - GET OEWS WAGE LINKS
# ============================================================

print("=" * 80)
print("STEP 1: GET OEWS WAGE HISTORY LINKS")
print("=" * 80)

all_links = []

main_html = fetch_html_no_crash(
    OEWS_TABLES_URL,
    SOURCE_PAGE_DIR / "oews_tables.htm"
)

if main_html:
    print(f"[{elapsed()}s] Extracting links from BLS source page...")

    main_links = extract_download_links_from_html(
        html=main_html,
        page_url=OEWS_TABLES_URL,
        page_label="OEWS main tables page"
    )

    all_links.extend(main_links)

    # Also try 1988-1995 historical page if found
    historical_page_urls = set()

    for href in re.findall(r'href=["\']([^"\']+)["\']', main_html, flags=re.IGNORECASE):
        full_url = urljoin(OEWS_TABLES_URL, href)

        if "estimates_88_95" in full_url:
            historical_page_urls.add(full_url)

    if not historical_page_urls:
        historical_page_urls.add(OEWS_1988_1995_URL)

    for historical_url in sorted(historical_page_urls):
        historical_html = fetch_html_no_crash(
            historical_url,
            SOURCE_PAGE_DIR / "estimates_88_95.htm"
        )

        if historical_html:
            historical_links = extract_download_links_from_html(
                html=historical_html,
                page_url=historical_url,
                page_label="OEWS 1988-1995 historical page"
            )

            all_links.extend(historical_links)

# If scrape failed or found too few links, use generated official URL list
if len(all_links) < 20:
    print()
    print("=" * 80)
    print("USING GENERATED OFFICIAL BLS DIRECT URL LIST")
    print("=" * 80)
    print("Reason: BLS page scrape was blocked or returned too few links.")
    print("This avoids the 403 error and continues downloading.")
    print()

    all_links = build_generated_oews_links()

# Deduplicate URLs
deduped_links = []
seen_urls = set()

for row in all_links:
    if row["url"] in seen_urls:
        continue

    seen_urls.add(row["url"])
    deduped_links.append(row)

all_links = deduped_links

print()
print("=" * 80)
print("OEWS LINK CHECK")
print("=" * 80)
print("Download mode:", DOWNLOAD_MODE)
print("OEWS wage/history links ready:", len(all_links))
print()

if len(all_links) == 0:
    raise RuntimeError("No OEWS links were created. Something is wrong with the link step.")

# Save link list
with open(SCRAPED_LINKS_FILE, "w", encoding="utf-8", newline="") as f:
    fieldnames = [
        "page_label",
        "period",
        "year_or_period",
        "category",
        "link_text",
        "url",
        "link_source"
    ]

    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_links)

print("Saved OEWS link list:")
print(SCRAPED_LINKS_FILE)
print()

print("First 15 links:")
for row in all_links[:15]:
    print(row["period"], "|", row["category"], "|", row["url"])

print()

# ============================================================
# 7. STEP 2 - DOWNLOAD PROJECTION / FORECAST FILE
# ============================================================

print("=" * 80)
print("STEP 2: DOWNLOAD PROJECTION / FORECAST FILE")
print("=" * 80)

manifest_rows = []

projection_result = download_file(
    url=PROJECTION_URL,
    save_path=PROJECTION_FILE,
    label="BLS occupation projection / forecast 2024-2034"
)

manifest_rows.append({
    "dataset_type": "projection_forecast",
    "period": "2024_2034",
    "category": "occupation_projection_forecast",
    "source_page": "https://www.bls.gov/emp/ind-occ-matrix/",
    "url": projection_result["url"],
    "saved_file": projection_result["file"],
    "status": projection_result["status"],
    "size_bytes": projection_result["size_bytes"],
    "error": projection_result["error"]
})

if projection_result["status"] == "failed":
    print()
    print("=" * 80)
    print("PROJECTION FILE FAILED")
    print("=" * 80)
    print("Manual browser download URL:")
    print(PROJECTION_URL)
    print()
    print("Save manually here:")
    print(PROJECTION_FILE)
    print()
    print("Then run this cell again. Existing files will be skipped.")
    print("=" * 80)
    print()

# ============================================================
# 8. STEP 3 - DOWNLOAD ALL OEWS WAGE HISTORY FILES
# ============================================================

print("=" * 80)
print("STEP 3: DOWNLOAD OEWS WAGE HISTORY FILES")
print("=" * 80)

used_names = set()

for i, row in enumerate(all_links, start=1):
    filename = make_download_filename(row, used_names)
    save_path = WAGE_DIR / filename

    result = download_file(
        url=row["url"],
        save_path=save_path,
        label=f"OEWS wage/history file {i}/{len(all_links)}"
    )

    manifest_rows.append({
        "dataset_type": "oews_wage_history",
        "period": row["period"],
        "category": row["category"],
        "source_page": OEWS_TABLES_URL,
        "url": result["url"],
        "saved_file": result["file"],
        "status": result["status"],
        "size_bytes": result["size_bytes"],
        "error": result["error"]
    })

    time.sleep(SLEEP_BETWEEN_DOWNLOADS)

# ============================================================
# 9. STEP 4 - SAVE DOWNLOAD MANIFEST
# ============================================================

print("=" * 80)
print("STEP 4: SAVE DOWNLOAD MANIFEST")
print("=" * 80)

with open(MANIFEST_FILE, "w", encoding="utf-8", newline="") as f:
    fieldnames = [
        "dataset_type",
        "period",
        "category",
        "source_page",
        "url",
        "saved_file",
        "status",
        "size_bytes",
        "error"
    ]

    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(manifest_rows)

print("Saved manifest:")
print(MANIFEST_FILE)
print()

# ============================================================
# 10. FINAL CHECK
# ============================================================

print("=" * 80)
print("FINAL DOWNLOAD CHECK")
print("=" * 80)

projection_files = [
    p for p in PROJECTION_DIR.glob("*")
    if p.is_file() and not p.name.endswith(".part")
]

wage_files = [
    p for p in WAGE_DIR.glob("*")
    if p.is_file() and not p.name.endswith(".part")
]

downloaded_count = sum(1 for r in manifest_rows if r["status"] == "downloaded")
skipped_count = sum(1 for r in manifest_rows if r["status"] == "skipped_exists")
failed_count = sum(1 for r in manifest_rows if r["status"] == "failed")

total_bytes = 0

for r in manifest_rows:
    try:
        total_bytes += int(r["size_bytes"])
    except Exception:
        pass

print("Projection / forecast folder:")
print(PROJECTION_DIR)
print("Projection files:", len(projection_files))
print()

print("OEWS wage history folder:")
print(WAGE_DIR)
print("OEWS wage history files:", len(wage_files))
print()

print("OEWS links used:")
print(len(all_links))
print()

print("Downloaded:", downloaded_count)
print("Skipped existing:", skipped_count)
print("Failed:", failed_count)
print(f"Total downloaded / existing size: {total_bytes / 1024 / 1024:.2f} MB")
print()

if failed_count > 0:
    print("=" * 80)
    print("SOME FILES FAILED")
    print("=" * 80)
    print("Check failed URLs here:")
    print(MANIFEST_FILE)
    print()
    print("This does not mean the whole project failed.")
    print("Downloaded files are still saved in:")
    print(WAGE_DIR)
    print()

print("=" * 80)
print("DONE - DOWNLOAD ONLY")
print("=" * 80)
print("Forecast file:")
print(PROJECTION_FILE)
print()
print("OEWS wage history files:")
print(WAGE_DIR)
print()
print("OEWS links log:")
print(SCRAPED_LINKS_FILE)
print()
print("Download manifest:")
print(MANIFEST_FILE)
print()
print(f"Total runtime: {elapsed()} seconds")

# Check

In [ ]:
from pathlib import Path
import shutil
import time

# ==================================================
# STEP 0: PUT BLS OCCUPATION FILE INTO RAW FOLDER
# Memory optimized:
# - Does NOT load Excel into pandas
# - Only checks/copies the file
# - Shows progress/status
# ==================================================

start = time.time()

PROJECT_DIR = Path.cwd()
RAW_DIR = PROJECT_DIR / "BLS_Job_Data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

target = RAW_DIR / "occupation_projection_2024_2034.xlsx"

possible_files = [
    PROJECT_DIR / "occupation.xlsx",
    PROJECT_DIR / "occupation_projection_2024_2034.xlsx",
    Path.home() / "Downloads" / "occupation.xlsx",
    Path.home() / "Downloads" / "occupation_projection_2024_2034.xlsx",
]

print("=" * 70)
print("CHECKING BLS OCCUPATION PROJECTION FILE")
print("=" * 70)
print("Current Jupyter folder:")
print(PROJECT_DIR)
print()
print("Required final file:")
print(target)
print()

# If final file already exists, no need to copy again
if target.exists() and target.stat().st_size > 0:
    size_mb = target.stat().st_size / (1024 * 1024)

    print("READY - file already exists in raw folder")
    print("File:", target)
    print(f"Size: {size_mb:,.2f} MB")

else:
    print("File not found in raw folder.")
    print("Searching possible locations...")
    print()

    found = None

    for file in possible_files:
        print("Checking:", file)

        if file.exists() and file.is_file() and file.stat().st_size > 0:
            found = file
            break

    print()

    if found:
        size_mb = found.stat().st_size / (1024 * 1024)

        print("FOUND FILE")
        print("From:", found)
        print(f"Size: {size_mb:,.2f} MB")
        print()

        print("Copying file into raw folder...")

        shutil.copy2(found, target)

        print("DONE - file copied")
        print("To:", target)

    else:
        print("NOT FOUND")
        print()
        print("Download this file manually:")
        print("https://www.bls.gov/emp/ind-occ-matrix/occupation.xlsx")
        print()
        print("Then save it as either:")
        print("occupation.xlsx")
        print()
        print("Put it in one of these places:")
        print("1.", PROJECT_DIR)
        print("2.", Path.home() / "Downloads")
        print("3.", RAW_DIR)
        print()
        print("Final file must be here:")
        print(target)

elapsed = time.time() - start
print()
print("=" * 70)
print(f"Finished in {elapsed:,.2f} seconds")
print("=" * 70)

In [ ]:
#!/usr/bin/env python3
"""
BLS OEWS/OES wage-history downloader - no-crash, memory-optimized version
Source page: https://www.bls.gov/oes/tables.htm

What this does:
1. Tries to scrape the BLS OEWS tables page.
2. If Python gets HTTP 403, tries curl automatically.
3. If BLS blocks scraping completely, uses a built-in official OEWS/OES URL list
   based on the BLS tables page structure, so the download step can still run.
4. Streams/downloads files without loading big files into RAM.
5. Never stops the whole project for one bad/blocked file.
6. Saves CSV reports for links, downloads, and failures.

Run in Jupyter:
    !python download_bls_oews_all_v2_no_error.py

Output:
    BLS_Job_Data/raw/oews_all_downloads/
    BLS_Job_Data/raw/oews_scraped_pages/
    BLS_Job_Data/output/oews_download_links.csv
    BLS_Job_Data/output/oews_download_report.csv
    BLS_Job_Data/output/oews_failed_downloads.csv
"""

from __future__ import annotations

import csv
import hashlib
import html
import os
import re
import shutil
import socket
import subprocess
import time
import traceback
from dataclasses import dataclass, asdict
from datetime import datetime
from html.parser import HTMLParser
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple
from urllib.error import HTTPError, URLError
from urllib.parse import urljoin, urlparse, urldefrag, unquote
from urllib.request import Request, urlopen

# =============================================================================
# USER SETTINGS
# =============================================================================

OEWS_TABLES_URL = "https://www.bls.gov/oes/tables.htm"
OES_1988_1995_URL = "https://www.bls.gov/oes/estimates_88_95.htm"
OEWS_TIME_SERIES_DIR = "https://download.bls.gov/pub/time.series/oe/"
SPECIAL_REQUESTS_BASE = "https://www.bls.gov/oes/special-requests/"

PROJECT_DIR = Path.cwd()
BASE_DIR = PROJECT_DIR / "BLS_Job_Data"
RAW_DIR = BASE_DIR / "raw"
OUTPUT_DIR = BASE_DIR / "output"
DOWNLOAD_DIR = RAW_DIR / "oews_all_downloads"
SCRAPED_PAGE_DIR = RAW_DIR / "oews_scraped_pages"

# Keep True because you said download everything.
DOWNLOAD_TIME_SERIES_TXT_FILES = True

# Keep True so the script still has files to download even when BLS blocks Python/curl HTML scraping.
ADD_BUILT_IN_BLS_FALLBACK_LINKS = True

# Keep True to find Research / Additional / Featured table links if BLS page scraping works.
FOLLOW_EXTRA_OEWS_DATA_PAGES = True

# Set to None for all files. For testing, set to 5, then set back to None.
MAX_FILES_TO_DOWNLOAD = None

# If False, already-downloaded files are skipped.
OVERWRITE_EXISTING = False

# Network / memory settings.
CHUNK_SIZE = 1024 * 1024       # 1 MB chunks = memory friendly
REQUEST_TIMEOUT = 120          # seconds
RETRIES = 3
SLEEP_BETWEEN_REQUESTS = 0.35
PROGRESS_EVERY_SECONDS = 5

# Prefer curl because BLS sometimes blocks Python urllib with 403.
USE_CURL_FIRST = True

DATA_EXTENSIONS = (
    ".xls", ".xlsx", ".xlsm", ".zip", ".txt", ".csv", ".tsv", ".dat"
)

ALLOWED_DOMAINS = {"www.bls.gov", "download.bls.gov"}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/605.1.15 (KHTML, like Gecko) "
        "Version/17.0 Safari/605.1.15"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "close",
}

# =============================================================================
# SMALL UTILITIES
# =============================================================================

def now_text() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def log(message: str) -> None:
    print(f"[{now_text()}] {message}", flush=True)


def ensure_dirs() -> None:
    for folder in [RAW_DIR, OUTPUT_DIR, DOWNLOAD_DIR, SCRAPED_PAGE_DIR]:
        folder.mkdir(parents=True, exist_ok=True)


def clean_url(url: str) -> str:
    url = html.unescape(url.strip())
    url, _frag = urldefrag(url)
    return url


def is_allowed_domain(url: str) -> bool:
    parsed = urlparse(url)
    return parsed.scheme in {"http", "https"} and parsed.netloc.lower() in ALLOWED_DOMAINS


def is_data_file_url(url: str) -> bool:
    path = unquote(urlparse(url).path).lower()
    basename = Path(path).name.lower()
    if path.endswith(DATA_EXTENSIONS):
        return True
    # BLS time-series files have names like oe.data.1.AllData with no normal extension.
    return parsed_is_oe_time_series(url) and basename.startswith("oe.")


def parsed_is_oe_time_series(url: str) -> bool:
    parsed = urlparse(url)
    return parsed.netloc.lower() == "download.bls.gov" and "/pub/time.series/oe/" in parsed.path


def is_probably_html(first_bytes: bytes) -> bool:
    sample = first_bytes[:500].lstrip().lower()
    return (
        sample.startswith(b"<!doctype html")
        or sample.startswith(b"<html")
        or b"<title>access denied" in sample
        or b"forbidden" in sample[:200]
    )


def short_hash(text: str, n: int = 10) -> str:
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()[:n]


def bytes_to_mb(n: Optional[int]) -> str:
    if n is None:
        return "unknown"
    return f"{n / (1024 * 1024):,.2f} MB"


def safe_filename_from_url(url: str) -> Path:
    parsed = urlparse(url)
    domain = parsed.netloc.lower().replace(":", "_")
    path = unquote(parsed.path).strip("/")

    if not path or path.endswith("/"):
        path = path.rstrip("/") + "/index.html"

    parts = []
    for part in path.split("/"):
        part = re.sub(r"[^A-Za-z0-9._()\-]+", "_", part)
        part = part.strip("._") or "blank"
        parts.append(part[:160])

    filename = Path(domain, *parts)

    if parsed.query:
        stem = filename.stem
        suffix = filename.suffix
        filename = filename.with_name(f"{stem}_{short_hash(parsed.query)}{suffix}")

    return filename


def infer_period_from_url_or_text(url: str, text: str = "") -> str:
    combined = f"{url} {text}"

    m = re.search(r"(?:May|November)\s+(19\d{2}|20\d{2})", combined, flags=re.I)
    if m:
        return m.group(0)

    m = re.search(r"oes([mn]?)(\d{2})", combined, flags=re.I)
    if m:
        month_flag = m.group(1).lower()
        yy = int(m.group(2))
        yyyy = 2000 + yy if yy <= 50 else 1900 + yy
        if month_flag == "m":
            return f"May {yyyy}"
        if month_flag == "n":
            return f"November {yyyy}"
        return str(yyyy)

    m = re.search(r"\b(1988|1989|199[0-9]|20[0-9]{2})\b", combined)
    if m:
        return m.group(1)

    if parsed_is_oe_time_series(url):
        return "current_time_series"

    return "unknown"


def get_file_type(url: str) -> str:
    parsed = urlparse(url)
    name = Path(unquote(parsed.path)).name
    suffix = Path(name).suffix.lower().lstrip(".")
    if suffix:
        return suffix
    if parsed_is_oe_time_series(url):
        return "bls_time_series_txt"
    return "no_extension"

# =============================================================================
# HTML LINK PARSER
# =============================================================================

@dataclass
class LinkRecord:
    source_page: str
    url: str
    link_text: str
    file_type: str
    period_guess: str
    discovery_method: str


class AnchorParser(HTMLParser):
    def __init__(self, base_url: str):
        super().__init__()
        self.base_url = base_url
        self.links: List[Tuple[str, str]] = []
        self._current_href: Optional[str] = None
        self._current_text: List[str] = []

    def handle_starttag(self, tag: str, attrs: List[Tuple[str, Optional[str]]]) -> None:
        if tag.lower() != "a":
            return
        attrs_dict = dict(attrs)
        href = attrs_dict.get("href")
        if href:
            self._current_href = href
            self._current_text = []

    def handle_data(self, data: str) -> None:
        if self._current_href is not None:
            self._current_text.append(data)

    def handle_endtag(self, tag: str) -> None:
        if tag.lower() == "a" and self._current_href is not None:
            text = " ".join(" ".join(self._current_text).split())
            abs_url = clean_url(urljoin(self.base_url, self._current_href))
            self.links.append((abs_url, text))
            self._current_href = None
            self._current_text = []


def parse_links(html_text: str, base_url: str) -> List[Tuple[str, str]]:
    parser = AnchorParser(base_url)
    parser.feed(html_text)
    return parser.links

# =============================================================================
# CURL HELPERS
# =============================================================================

def has_curl() -> bool:
    return shutil.which("curl") is not None


def curl_download_to_file(url: str, dest: Path, show_progress: bool) -> Tuple[bool, str, int]:
    """Use curl to download URL to dest. Memory-friendly because curl writes to disk."""
    if not has_curl():
        return False, "curl not found", 0

    dest.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        "curl",
        "--location",
        "--fail",
        "--retry", str(RETRIES),
        "--retry-delay", "3",
        "--connect-timeout", "30",
        "--max-time", str(REQUEST_TIMEOUT * 3),
        "--user-agent", HEADERS["User-Agent"],
        "--header", "Accept: */*",
        "--output", str(dest),
        url,
    ]

    if show_progress:
        cmd.insert(1, "--progress-bar")
    else:
        cmd.insert(1, "--silent")
        cmd.insert(2, "--show-error")

    try:
        proc = subprocess.run(cmd, text=True, stdout=None if show_progress else subprocess.PIPE,
                              stderr=None if show_progress else subprocess.PIPE)
        if proc.returncode == 0 and dest.exists() and dest.stat().st_size > 0:
            return True, "OK", proc.returncode
        message = "curl failed"
        if not show_progress:
            message = (proc.stderr or proc.stdout or message).strip()[:500]
        return False, message, proc.returncode
    except Exception as exc:
        return False, f"curl exception: {type(exc).__name__}: {exc}", 0


def curl_fetch_html(url: str, dest: Path) -> Optional[str]:
    tmp = dest.with_name(dest.name + ".curl.part")
    if tmp.exists():
        try:
            tmp.unlink()
        except Exception:
            pass

    for attempt in range(1, RETRIES + 1):
        log(f"Fetching HTML with curl attempt {attempt}/{RETRIES}: {url}")
        ok, message, code = curl_download_to_file(url, tmp, show_progress=False)
        if ok:
            data = tmp.read_bytes()
            if is_probably_html(data):
                tmp.replace(dest)
                text = dest.read_text(encoding="utf-8", errors="replace")
                log(f"Saved scraped HTML by curl: {dest}")
                return text
            log("curl returned content, but it does not look like HTML.")
        else:
            log(f"curl HTML fetch failed: code={code}, {message}")
        time.sleep(2 * attempt)

    try:
        if tmp.exists():
            tmp.unlink()
    except Exception:
        pass
    return None

# =============================================================================
# PYTHON NETWORK FUNCTIONS
# =============================================================================

def fetch_bytes_python(url: str, timeout: int = REQUEST_TIMEOUT) -> Tuple[bytes, Dict[str, str]]:
    req = Request(url, headers=HEADERS)
    with urlopen(req, timeout=timeout) as response:
        headers = dict(response.headers.items())
        data = response.read()
    return data, headers


def fetch_html_page(url: str) -> Optional[str]:
    """Fetch HTML page and save it. Return text or None if blocked/failed."""
    local_name = safe_filename_from_url(url)
    local_path = SCRAPED_PAGE_DIR / local_name
    local_path.parent.mkdir(parents=True, exist_ok=True)

    # Use existing local copy first only if it is not empty.
    if local_path.exists() and local_path.stat().st_size > 0:
        log(f"Using existing local HTML backup: {local_path}")
        return local_path.read_text(encoding="utf-8", errors="replace")

    if USE_CURL_FIRST:
        text = curl_fetch_html(url, local_path)
        if text:
            return text

    for attempt in range(1, RETRIES + 1):
        try:
            log(f"Fetching HTML with Python attempt {attempt}/{RETRIES}: {url}")
            data, headers = fetch_bytes_python(url)
            encoding = "utf-8"
            content_type = headers.get("Content-Type", "")
            m = re.search(r"charset=([A-Za-z0-9_\-]+)", content_type, flags=re.I)
            if m:
                encoding = m.group(1)
            text = data.decode(encoding, errors="replace")
            local_path.write_text(text, encoding="utf-8")
            log(f"Saved scraped HTML by Python: {local_path}")
            return text
        except Exception as exc:
            log(f"Python HTML fetch failed attempt {attempt}: {type(exc).__name__}: {exc}")
            time.sleep(2 * attempt)

    log(f"Could not fetch HTML and no local backup exists: {url}")
    return None


def get_content_length(headers: Dict[str, str]) -> Optional[int]:
    value = headers.get("Content-Length") or headers.get("content-length")
    if not value:
        return None
    try:
        return int(value)
    except ValueError:
        return None

# =============================================================================
# DOWNLOAD FUNCTIONS
# =============================================================================

def validate_downloaded_file(path: Path, url: str) -> Tuple[bool, str]:
    if not path.exists():
        return False, "file does not exist"
    size = path.stat().st_size
    if size <= 0:
        return False, "file is empty"
    # Check first bytes only; memory safe.
    with open(path, "rb") as f:
        first = f.read(800)
    if is_data_file_url(url) and is_probably_html(first):
        return False, "downloaded HTML instead of data file; likely blocked/missing"
    return True, "OK"


def stream_download_python(url: str, dest: Path) -> Dict[str, object]:
    started = time.time()
    dest.parent.mkdir(parents=True, exist_ok=True)
    part_path = dest.with_name(dest.name + ".part")

    result = {
        "url": url,
        "local_path": str(dest),
        "status": "failed",
        "method": "python_stream",
        "http_status": "",
        "bytes_downloaded": 0,
        "size_mb": 0.0,
        "seconds": 0.0,
        "message": "",
    }

    if part_path.exists():
        try:
            part_path.unlink()
        except Exception:
            pass

    for attempt in range(1, RETRIES + 1):
        try:
            log(f"Downloading with Python stream attempt {attempt}/{RETRIES}: {url}")
            req = Request(url, headers=HEADERS)
            with urlopen(req, timeout=REQUEST_TIMEOUT) as response:
                http_status = getattr(response, "status", "")
                headers = dict(response.headers.items())
                total_size = get_content_length(headers)
                result["http_status"] = http_status

                downloaded = 0
                last_print = time.time()
                first_chunk = response.read(CHUNK_SIZE)

                if is_data_file_url(url) and is_probably_html(first_chunk):
                    raise RuntimeError("Expected data file but server returned HTML page, possibly blocked or missing.")

                with open(part_path, "wb") as f:
                    if first_chunk:
                        f.write(first_chunk)
                        downloaded += len(first_chunk)

                    while True:
                        chunk = response.read(CHUNK_SIZE)
                        if not chunk:
                            break
                        f.write(chunk)
                        downloaded += len(chunk)

                        now = time.time()
                        if now - last_print >= PROGRESS_EVERY_SECONDS:
                            if total_size:
                                pct = downloaded / total_size * 100
                                log(f"  progress: {bytes_to_mb(downloaded)} / {bytes_to_mb(total_size)} ({pct:.1f}%)")
                            else:
                                log(f"  progress: {bytes_to_mb(downloaded)}")
                            last_print = now

            part_path.replace(dest)
            valid, msg = validate_downloaded_file(dest, url)
            if not valid:
                try:
                    dest.unlink()
                except Exception:
                    pass
                raise RuntimeError(msg)

            elapsed = time.time() - started
            result.update({
                "status": "downloaded",
                "bytes_downloaded": dest.stat().st_size,
                "size_mb": round(dest.stat().st_size / (1024 * 1024), 3),
                "seconds": round(elapsed, 2),
                "message": "OK",
            })
            log(f"DONE: {dest} ({bytes_to_mb(dest.stat().st_size)})")
            return result

        except HTTPError as exc:
            result["http_status"] = exc.code
            result["message"] = f"HTTPError {exc.code}: {exc.reason}"
            log(f"Python download failed: {result['message']}")
        except (URLError, socket.timeout, TimeoutError) as exc:
            result["message"] = f"Network error: {type(exc).__name__}: {exc}"
            log(f"Python download failed: {result['message']}")
        except Exception as exc:
            result["message"] = f"{type(exc).__name__}: {exc}"
            log(f"Python download failed: {result['message']}")

        try:
            if part_path.exists():
                part_path.unlink()
        except Exception:
            pass
        time.sleep(3 * attempt)

    result["seconds"] = round(time.time() - started, 2)
    return result


def download_one_file(url: str, dest: Path) -> Dict[str, object]:
    started = time.time()
    result = {
        "url": url,
        "local_path": str(dest),
        "status": "failed",
        "method": "",
        "http_status": "",
        "bytes_downloaded": 0,
        "size_mb": 0.0,
        "seconds": 0.0,
        "message": "",
    }

    if dest.exists() and not OVERWRITE_EXISTING:
        valid, msg = validate_downloaded_file(dest, url)
        if valid:
            size = dest.stat().st_size
            result.update({
                "status": "skipped_existing",
                "method": "existing_file",
                "bytes_downloaded": size,
                "size_mb": round(size / (1024 * 1024), 3),
                "seconds": round(time.time() - started, 2),
                "message": "File already exists; skipped. Set OVERWRITE_EXISTING=True to redownload.",
            })
            return result
        log(f"Existing file is invalid ({msg}); redownloading: {dest}")
        try:
            dest.unlink()
        except Exception:
            pass

    # First try curl because it often succeeds when urllib gets BLS 403.
    if USE_CURL_FIRST and has_curl():
        part = dest.with_name(dest.name + ".curl.part")
        if part.exists():
            try:
                part.unlink()
            except Exception:
                pass

        log(f"Downloading with curl: {url}")
        ok, message, code = curl_download_to_file(url, part, show_progress=True)
        if ok:
            valid, msg = validate_downloaded_file(part, url)
            if valid:
                part.replace(dest)
                size = dest.stat().st_size
                result.update({
                    "status": "downloaded",
                    "method": "curl",
                    "http_status": "",
                    "bytes_downloaded": size,
                    "size_mb": round(size / (1024 * 1024), 3),
                    "seconds": round(time.time() - started, 2),
                    "message": "OK",
                })
                log(f"DONE: {dest} ({bytes_to_mb(size)})")
                return result
            message = msg
            try:
                part.unlink()
            except Exception:
                pass
        else:
            try:
                if part.exists():
                    part.unlink()
            except Exception:
                pass
        log(f"curl failed, will try Python stream next: code={code}, {message}")
        result["method"] = "curl_then_python"
        result["message"] = f"curl failed: {message}"

    py_result = stream_download_python(url, dest)
    if py_result.get("status") == "downloaded":
        return py_result

    # Return the more useful combined message.
    if result["message"]:
        py_result["message"] = f"{result['message']} | Python: {py_result.get('message', '')}"
        py_result["method"] = "curl_then_python"
    return py_result

# =============================================================================
# SCRAPE LOGIC
# =============================================================================

def should_follow_extra_page(url: str, link_text: str) -> bool:
    if not FOLLOW_EXTRA_OEWS_DATA_PAGES:
        return False

    parsed = urlparse(url)
    if parsed.netloc.lower() != "www.bls.gov":
        return False
    if not parsed.path.lower().startswith("/oes/"):
        return False

    text = f"{link_text} {parsed.path}".lower()
    wanted = [
        "1988", "1995", "estimates_88_95",
        "research estimates", "research", "additional oews", "additional",
        "featured tables", "featured", "mb3",
    ]
    return any(key in text for key in wanted)


def discover_links_from_page(page_url: str, html_text: str) -> Tuple[List[LinkRecord], List[str]]:
    data_links: List[LinkRecord] = []
    extra_pages: List[str] = []

    for url, text in parse_links(html_text, page_url):
        if not is_allowed_domain(url):
            continue

        if is_data_file_url(url):
            data_links.append(LinkRecord(
                source_page=page_url,
                url=url,
                link_text=text,
                file_type=get_file_type(url),
                period_guess=infer_period_from_url_or_text(url, text),
                discovery_method="scraped_html",
            ))
        elif should_follow_extra_page(url, text):
            extra_pages.append(url)

    return data_links, sorted(set(extra_pages))


def add_record(records: List[LinkRecord], source_page: str, url: str, text: str, method: str) -> None:
    records.append(LinkRecord(
        source_page=source_page,
        url=url,
        link_text=text,
        file_type=get_file_type(url),
        period_guess=infer_period_from_url_or_text(url, text),
        discovery_method=method,
    ))


def built_in_bls_fallback_links() -> List[LinkRecord]:
    """Official BLS OEWS/OES URL patterns from the OEWS tables page.

    This fallback is used because BLS sometimes returns HTTP 403 to Python scraping.
    It covers the main downloadable table files on https://www.bls.gov/oes/tables.htm
    plus the 1988-1995 estimates and the OEWS time-series files.
    """
    records: List[LinkRecord] = []
    method = "built_in_official_bls_url_pattern"

    def sr(filename: str, label: str, source: str = OEWS_TABLES_URL) -> None:
        add_record(records, source, urljoin(SPECIAL_REQUESTS_BASE, filename), label, method)

    # May 2025-2012: standard 5-file OEWS set.
    for year in range(2025, 2011, -1):
        yy = str(year)[-2:]
        for code, label in [
            ("nat", "National"),
            ("st", "State"),
            ("ma", "Metropolitan and nonmetropolitan area"),
            ("ind", "National industry-specific and by ownership"),
            ("all", "All data"),
        ]:
            sr(f"oesm{yy}{code}.zip", f"May {year} {label}")

    # November 2011 agricultural sector special file.
    add_record(records, OEWS_TABLES_URL, "https://www.bls.gov/oes/ag_data_2011.xlsx",
               "November 2011 Agricultural Sector Estimates", method)

    # May 2011: standard 5-file set.
    for code, label in [
        ("nat", "National"), ("st", "State"), ("ma", "Metropolitan and nonmetropolitan area"),
        ("ind", "National industry-specific and by ownership"), ("all", "All data"),
    ]:
        sr(f"oesm11{code}.zip", f"May 2011 {label}")

    # May 2010-2005: 4-file set; industry file uses in4.
    for year in range(2010, 2004, -1):
        yy = str(year)[-2:]
        for code, label in [
            ("nat", "National"),
            ("st", "State"),
            ("ma", "Metropolitan/metropolitan and nonmetropolitan area"),
            ("in4", "National industry-specific"),
        ]:
            sr(f"oesm{yy}{code}.zip", f"May {year} {label}")

    # November 2004 and May 2004.
    for prefix, period in [("oesn04", "November 2004"), ("oesm04", "May 2004")]:
        for code, label in [("nat", "National"), ("st", "State"), ("ma", "Metropolitan area"), ("in4", "National industry-specific")]:
            sr(f"{prefix}{code}.zip", f"{period} {label}")

    # November 2003 and May 2003.
    for prefix, period in [("oesn03", "November 2003"), ("oesm03", "May 2003")]:
        for code, label in [("nat", "National"), ("st", "State"), ("ma", "Metropolitan area"), ("in4", "National industry-specific")]:
            sr(f"{prefix}{code}.zip", f"{period} {label}")

    # 2002 uses industry in4.
    for code, label in [("nat", "National"), ("st", "State"), ("ma", "Metropolitan area"), ("in4", "National industry-specific")]:
        sr(f"oes02{code}.zip", f"2002 {label}")

    # 2001-1997 use industry in3.
    for year in range(2001, 1996, -1):
        yy = str(year)[-2:]
        for code, label in [("nat", "National"), ("st", "State"), ("ma", "Metropolitan area"), ("in3", "National industry-specific")]:
            sr(f"oes{yy}{code}.zip", f"{year} {label}")

    # 1988-1995 OES estimates page: 2-digit and 3-digit SIC XLS files.
    old_files = [
        (1995, "mf95d2.xls", "2-digit"), (1995, "mf95d3.xls", "3-digit"),
        (1994, "bn94d2.xls", "2-digit"), (1994, "bn94d3.xls", "3-digit"),
        (1993, "nm93d2.xls", "2-digit"), (1993, "nm93d3.xls", "3-digit"),
        (1992, "mf92d2.xls", "2-digit"), (1992, "mf92d3.xls", "3-digit"),
        (1991, "bn91d2.xls", "2-digit"), (1991, "bn91d3.xls", "3-digit"),
        (1990, "nm90d2.xls", "2-digit"), (1990, "nm90d3.xls", "3-digit"),
        (1989, "mf89d2.xls", "2-digit"), (1989, "mf89d3.xls", "3-digit"),
        (1988, "bn88d2.xls", "2-digit"), (1988, "bn88d3.xls", "3-digit"),
    ]
    for year, filename, level in old_files:
        sr(filename, f"{year} OES estimate {level}", source=OES_1988_1995_URL)

    # OEWS time-series directory files. These may be large but are plain BLS text files.
    if DOWNLOAD_TIME_SERIES_TXT_FILES:
        for filename in [
            "oe.area", "oe.areatype", "oe.contacts", "oe.data.0.Current", "oe.data.1.AllData",
            "oe.datatype", "oe.footnote", "oe.industry", "oe.occupation", "oe.release",
            "oe.seasonal", "oe.sector", "oe.series", "oe.txt",
        ]:
            add_record(records, OEWS_TIME_SERIES_DIR, urljoin(OEWS_TIME_SERIES_DIR, filename),
                       f"OEWS time-series file {filename}", method)

    return records


def discover_time_series_files_by_scrape() -> List[LinkRecord]:
    if not DOWNLOAD_TIME_SERIES_TXT_FILES:
        return []

    html_text = fetch_html_page(OEWS_TIME_SERIES_DIR)
    if not html_text:
        return []

    records: List[LinkRecord] = []
    for url, text in parse_links(html_text, OEWS_TIME_SERIES_DIR):
        if not is_allowed_domain(url):
            continue
        if not parsed_is_oe_time_series(url):
            continue
        basename = Path(unquote(urlparse(url).path)).name
        if not basename:
            continue
        if basename.startswith("[") or "parent" in text.lower():
            continue
        if basename.startswith("oe."):
            records.append(LinkRecord(
                source_page=OEWS_TIME_SERIES_DIR,
                url=url,
                link_text=text or basename,
                file_type="bls_time_series_txt",
                period_guess="current_time_series",
                discovery_method="scraped_time_series_directory",
            ))
    return records


def dedupe_records(records: Iterable[LinkRecord]) -> List[LinkRecord]:
    seen = set()
    out = []
    for rec in records:
        if rec.url in seen:
            continue
        seen.add(rec.url)
        out.append(rec)
    return out


def write_csv(path: Path, rows: Iterable[Dict[str, object]], fieldnames: List[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def build_download_path(url: str) -> Path:
    return DOWNLOAD_DIR / safe_filename_from_url(url)

# =============================================================================
# MAIN
# =============================================================================

def main() -> int:
    ensure_dirs()
    start = time.time()
    log("STARTING BLS OEWS / OES DOWNLOAD PROJECT")
    log(f"Project folder: {PROJECT_DIR}")
    log(f"Raw download folder: {DOWNLOAD_DIR}")
    log("MEMORY OPTIMIZED: streaming/curl downloads; no pandas; no full-file loading.")
    log("NO-CRASH MODE: blocked/missing files go to CSV report; project continues.")

    all_records: List[LinkRecord] = []
    pages_to_scrape = [OEWS_TABLES_URL, OES_1988_1995_URL]
    scraped_pages = set()

    index = 0
    while index < len(pages_to_scrape):
        page_url = pages_to_scrape[index]
        index += 1
        if page_url in scraped_pages:
            continue
        scraped_pages.add(page_url)

        html_text = fetch_html_page(page_url)
        if not html_text:
            continue

        records, extra_pages = discover_links_from_page(page_url, html_text)
        all_records.extend(records)

        for extra in extra_pages:
            if extra not in scraped_pages and extra not in pages_to_scrape:
                pages_to_scrape.append(extra)

        time.sleep(SLEEP_BETWEEN_REQUESTS)

    # Add time-series from directory scrape if possible.
    all_records.extend(discover_time_series_files_by_scrape())

    scraped_count = len(dedupe_records(all_records))
    log(f"Scraped downloadable links found: {scraped_count:,}")

    # Important: always add fallback links, then dedupe. This solves BLS 403 page blocking.
    if ADD_BUILT_IN_BLS_FALLBACK_LINKS:
        fallback = built_in_bls_fallback_links()
        log(f"Adding built-in official BLS fallback links: {len(fallback):,}")
        all_records.extend(fallback)

    all_records = dedupe_records(all_records)
    all_records.sort(key=lambda r: (r.period_guess, r.file_type, r.url))

    links_csv = OUTPUT_DIR / "oews_download_links.csv"
    write_csv(
        links_csv,
        [asdict(r) for r in all_records],
        ["source_page", "url", "link_text", "file_type", "period_guess", "discovery_method"],
    )
    log(f"Total downloadable links to try: {len(all_records):,}")
    log(f"Saved link list: {links_csv}")

    if not all_records:
        log("No downloadable files were discovered or generated. Finished without crash.")
        return 0

    records_to_download = all_records
    if MAX_FILES_TO_DOWNLOAD is not None:
        records_to_download = all_records[:MAX_FILES_TO_DOWNLOAD]
        log(f"TEST MODE: only downloading first {MAX_FILES_TO_DOWNLOAD} files.")

    report_rows: List[Dict[str, object]] = []
    total = len(records_to_download)

    report_fields = [
        "number", "source_page", "url", "link_text", "file_type", "period_guess", "discovery_method",
        "local_path", "status", "method", "http_status", "bytes_downloaded", "size_mb", "seconds", "message",
    ]

    report_csv = OUTPUT_DIR / "oews_download_report.csv"
    failed_csv = OUTPUT_DIR / "oews_failed_downloads.csv"

    for i, rec in enumerate(records_to_download, start=1):
        log("=" * 80)
        log(f"FILE {i:,}/{total:,}: {rec.url}")
        dest = build_download_path(rec.url)
        try:
            result = download_one_file(rec.url, dest)
        except Exception as exc:
            # Extra safety net so one file never kills the job.
            result = {
                "url": rec.url,
                "local_path": str(dest),
                "status": "failed",
                "method": "safety_net",
                "http_status": "",
                "bytes_downloaded": 0,
                "size_mb": 0.0,
                "seconds": 0.0,
                "message": f"Unexpected per-file error: {type(exc).__name__}: {exc}",
            }
            log(result["message"])

        row = {
            "number": i,
            "source_page": rec.source_page,
            "url": rec.url,
            "link_text": rec.link_text,
            "file_type": rec.file_type,
            "period_guess": rec.period_guess,
            "discovery_method": rec.discovery_method,
            **result,
        }
        report_rows.append(row)

        # Save progress after every file so you do not lose report if interrupted.
        write_csv(report_csv, report_rows, report_fields)
        failed_rows_live = [r for r in report_rows if r.get("status") == "failed"]
        write_csv(failed_csv, failed_rows_live, report_fields)

        time.sleep(SLEEP_BETWEEN_REQUESTS)

    downloaded = sum(1 for r in report_rows if r.get("status") == "downloaded")
    skipped = sum(1 for r in report_rows if r.get("status") == "skipped_existing")
    failed = sum(1 for r in report_rows if r.get("status") == "failed")
    total_mb = sum(float(r.get("size_mb") or 0) for r in report_rows)

    log("=" * 80)
    log("FINISHED")
    log(f"Downloaded: {downloaded:,}")
    log(f"Skipped existing: {skipped:,}")
    log(f"Failed but not crashed: {failed:,}")
    log(f"Total downloaded/skipped size in report: {total_mb:,.2f} MB")
    log(f"Links CSV: {links_csv}")
    log(f"Report CSV: {report_csv}")
    log(f"Failed CSV: {failed_csv}")
    log(f"Elapsed: {(time.time() - start) / 60:,.2f} minutes")

    if failed:
        log("Some links failed because BLS may block downloads, move old files, or timeout.")
        log("This is not a code crash. Check oews_failed_downloads.csv for exact URLs.")
        log("Run the script again later; completed files will be skipped automatically.")

    return 0


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        log("Stopped by user. Completed files are kept. Run again to continue; existing files will be skipped.")
    except Exception as exc:
        # Last safety net: do not show a long traceback in Jupyter unless user opens the log.
        ensure_dirs()
        error_log = OUTPUT_DIR / "oews_unexpected_error.txt"
        error_log.write_text(
            f"Unexpected error at {now_text()}\n{type(exc).__name__}: {exc}\n\n{traceback.format_exc()}",
            encoding="utf-8",
        )
        log(f"Unexpected error saved to: {error_log}")
        log(f"Error: {type(exc).__name__}: {exc}")
        log("Finished with error captured in log file, not a notebook traceback.")


# Fail just manually download: 
# https://www.bls.gov/oes/tables.htm

| On BLS page                         |      Download? | Why                                                                                                                                                                                                 |
| ----------------------------------- | -------------: | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **National XLSX**                   |          ✅ Yes | Best for your job forecast + wage history project                                                                                                                                                   |
| **National HTML**                   |           ❌ No | Same information for viewing online, not for data cleaning                                                                                                                                          |
| **State XLSX**                      | ❌ Skip for now | Only needed if you want wage by state                                                                                                                                                               |
| **Metro / nonmetro XLSX**           | ❌ Skip for now | Only needed if you want city/metro wage                                                                                                                                                             |
| **National industry-specific XLSX** | ❌ Skip for now | More detailed by industry; can make project bigger and harder                                                                                                                                       |
| **All data XLSX**                   |    ⚠️ Optional | Big file; includes more than you need if you only want national occupation wage history                                                                                                             |
| **All data TXT**                    |         ❌ Skip | Huge time-series text files; not needed for manual Excel download. The TXT directory has very large files, including `oe.data.0.Current`, `oe.data.1.AllData`, and `oe.series`. ([BLS Download][1]) |
| **Occupational Profiles**           |         ❌ Skip | Web profile pages, not your clean raw data                                                                                                                                                          |
| **Research estimates**              |         ❌ Skip | Special research data, not regular OEWS wage history                                                                                                                                                |
| **Additional OEWS data sets**       |         ❌ Skip | Extra datasets, not needed now                                                                                                                                                                      |
| **OEWS Maps / Featured Tables**     |         ❌ Skip | Display/summary tools, not raw history data                                                                                                                                                         |

[1]: https://download.bls.gov/pub/time.series/oe/ "download.bls.gov - /pub/time.series/oe/"


| Listed item                                     |                   Is it inside All data? | Need separate download? |
| ----------------------------------------------- | ---------------------------------------: | ----------------------: |
| **National**                                    |                                      Yes |                      No |
| **State**                                       |                                      Yes |                      No |
| **Metropolitan and nonmetropolitan area**       |                                      Yes |                      No |
| **National industry-specific and by ownership** |                                      Yes |                      No |
| **All data TXT**                                | Same type of data, different file format |     No, if you use XLSX |


| Item                          | In All data? | Meaning                                                                                      |
| ----------------------------- | -----------: | -------------------------------------------------------------------------------------------- |
| **Occupational Profiles**     |           No | Separate profile/HTML pages. Good for browsing, not needed for your main dataset.            |
| **Research estimates**        |           No | Extra research-only estimates, like state + industry or state + ownership. Separate dataset. |
| **Additional OEWS data sets** |           No | Extra/special datasets, not the main yearly wage file.                                       |
| **OEWS Maps**                 |           No | Website map/visual pages, not needed for your dataset.                                       |
| **Featured Tables**           |           No | Pre-made summary tables/articles, not needed if you have All data.                           |


| Link                          | What it is for                                                                                                                                                                                                | Do you need it? |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | --------------: |
| **Occupational Profiles**     | Web pages for each occupation. Good for clicking and reading job pages. It shows occupation definition, national wage/employment, industry profile, and geographic profile. ([Bureau of Labor Statistics][1]) |      Usually no |
| **Research estimates**        | Extra research-only data by **state + industry** or **state + ownership**. BLS says to use caution because the estimates have more limitations and fewer quality checks. ([Bureau of Labor Statistics][2])    |        Optional |
| **Additional OEWS data sets** | Extra special files, like **STEM data** and **typical entry-level education** files. ([Bureau of Labor Statistics][3])                                                                                        |    Maybe useful |
| **OEWS Maps**                 | Interactive map website for viewing wages/employment by location. ([Bureau of Labor Statistics][4])                                                                                                           |              No |
| **Featured Tables**           | Summary/overview tables, like largest occupations and major occupational groups. ([Bureau of Labor Statistics][5])                                                                                            |              No |

[1]: https://www.bls.gov/oes/2025/may/oes_stru.htm "May 2025 Occupation Profiles"
[2]: https://www.bls.gov/oes/oessrcres.htm "OEWS Research Estimates by State and Industry or State and Ownership :  U.S. Bureau of Labor Statistics"
[3]: https://www.bls.gov/oes/additional.htm "Additional OEWS data sets :  U.S. Bureau of Labor Statistics"
[4]: https://data.bls.gov/oesmap/ "Occupational Employment and Wage Statistics Maps"
[5]: https://www.bls.gov/oes/2025/may/overview_2025.htm "Overview of May 2025 Occupational Employment and Wages"
